# Solar Filament Segmentation 2026 — U-Net GPU

Notebook para treinar uma U-Net binária no Kaggle com GPU, usando as máscaras COCO do MAGFiLO. O modelo prevê a união dos filamentos por imagem; após a inferência, componentes conectados são convertidos em instâncias e codificados em RLE COCO para gerar `submission.csv`.

**Configuração recomendada:** Kaggle Notebook → Settings → Accelerator → GPU.

In [ ]:
import os, json, random, math, csv
from pathlib import Path
import numpy as np
from PIL import Image, ImageEnhance
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pycocotools import mask as mask_utils
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects

print('torch:', torch.__version__)
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Ative GPU em Settings > Accelerator > GPU'
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda'
# Localiza automaticamente o dataset, mesmo que o Kaggle use outro nome de pasta.
INPUT=Path('/kaggle/input')
json_candidates=sorted(INPUT.rglob('*.json'))
ann_candidates=[p for p in json_candidates if 'train' in str(p).lower() and 'annotation' in p.name.lower()]
if not ann_candidates:
    ann_candidates=[p for p in json_candidates if 'train' in str(p).lower()]
if not ann_candidates:
    raise FileNotFoundError('Nenhum JSON de anotação encontrado. No Kaggle, use Add Input e anexe os dados da competição.')
ANN_PATH=ann_candidates[0]
train_candidates=[p for p in INPUT.rglob('train_images') if p.is_dir()]
test_candidates=[p for p in INPUT.rglob('test_images') if p.is_dir()]
if not train_candidates or not test_candidates:
    raise FileNotFoundError('Pastas train_images/test_images não encontradas. Anexe o dataset completo da competição em Add Input.')
TRAIN_DIR=train_candidates[0]
TEST_DIR=test_candidates[0]
DATA=ANN_PATH.parent
OUT=Path('/kaggle/working')
print('annotation:', ANN_PATH)
print('train images:', TRAIN_DIR)
print('test images:', TEST_DIR)

## 1. COCO annotations and image-level masks

In [ ]:
coco=json.loads(ANN_PATH.read_text())
images={str(x['id']):x for x in coco['images']}
by_file={}
for ann in coco['annotations']:
    by_file.setdefault(images[str(ann['image_id'])]['file_name'],[]).append(ann)
files=sorted([p.name for p in TRAIN_DIR.glob('*.jpeg') if p.name in by_file])
print('train files:',len(files),'test files:',len(list(TEST_DIR.glob('*.jpeg'))))

def get_mask(name):
    out=np.zeros((2048,2048),np.uint8)
    for ann in by_file.get(name,[]):
        rr=mask_utils.frPyObjects(ann['segmentation'],2048,2048)
        out |= mask_utils.decode(mask_utils.merge(rr)).astype(np.uint8)
    return out

In [ ]:
random.Random(SEED).shuffle(files)
cut=int(0.85*len(files)); train_files=files[:cut]; valid_files=files[cut:]
IMG_SIZE=512
class FilamentDataset(Dataset):
    def __init__(self, names, augment=False): self.names=names; self.augment=augment
    def __len__(self): return len(self.names)
    def __getitem__(self,i):
        name=self.names[i]
        x=np.asarray(Image.open(TRAIN_DIR/name).convert('L'),np.float32)/255.0
        y=get_mask(name).astype(np.float32)
        # resize to 512x512 to fit a free Kaggle GPU; masks stay binary
        x=np.asarray(Image.fromarray((x*255).astype(np.uint8)).resize((IMG_SIZE,IMG_SIZE),Image.BILINEAR),np.float32)/255.0
        y=np.asarray(Image.fromarray((y*255).astype(np.uint8)).resize((IMG_SIZE,IMG_SIZE),Image.NEAREST),np.float32)/255.0
        if self.augment:
            if random.random()<0.5: x=x[:,::-1].copy(); y=y[:,::-1].copy()
            if random.random()<0.5: x=x[::-1,:].copy(); y=y[::-1,:].copy()
            if random.random()<0.5: x=np.clip(x*random.uniform(0.85,1.15)+random.uniform(-0.06,0.06),0,1)
        return torch.from_numpy(x[None]), torch.from_numpy(y[None])
train_dl=DataLoader(FilamentDataset(train_files,True),batch_size=8,shuffle=True,num_workers=2,pin_memory=True)
valid_dl=DataLoader(FilamentDataset(valid_files),batch_size=8,shuffle=False,num_workers=2,pin_memory=True)

## 2. U-Net, loss, and metrics

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, a,b):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True),nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
    def forward(self,x): return self.net(x)
class UNet(nn.Module):
    def __init__(self):
        super().__init__(); c=[32,64,128,256]
        self.e1=DoubleConv(1,c[0]); self.e2=DoubleConv(c[0],c[1]); self.e3=DoubleConv(c[1],c[2]); self.e4=DoubleConv(c[2],c[3]); self.pool=nn.MaxPool2d(2)
        self.mid=DoubleConv(c[3],512); self.u4=nn.ConvTranspose2d(512,c[3],2,2); self.d4=DoubleConv(c[3]*2,c[3]); self.u3=nn.ConvTranspose2d(c[3],c[2],2,2); self.d3=DoubleConv(c[2]*2,c[2]); self.u2=nn.ConvTranspose2d(c[2],c[1],2,2); self.d2=DoubleConv(c[1]*2,c[1]); self.u1=nn.ConvTranspose2d(c[1],c[0],2,2); self.d1=DoubleConv(c[0]*2,c[0]); self.out=nn.Conv2d(c[0],1,1)
    def forward(self,x):
        a=self.e1(x); b=self.e2(self.pool(a)); c=self.e3(self.pool(b)); d=self.e4(self.pool(c)); z=self.mid(self.pool(d))
        z=self.d4(torch.cat([self.u4(z),d],1)); z=self.d3(torch.cat([self.u3(z),c],1)); z=self.d2(torch.cat([self.u2(z),b],1)); z=self.d1(torch.cat([self.u1(z),a],1)); return self.out(z)

def dice_loss(logit,y):
    p=torch.sigmoid(logit); inter=(p*y).sum((1,2,3)); return (1-(2*inter+1)/(p.sum((1,2,3))+y.sum((1,2,3))+1)).mean()

def loss_fn(logit,y): return 0.5*F.binary_cross_entropy_with_logits(logit,y)+0.5*dice_loss(logit,y)

def dice_metric(logit,y):
    p=(torch.sigmoid(logit)>0.5).float(); inter=(p*y).sum(); return float((2*inter+1)/(p.sum()+y.sum()+1))

## 3. Train with mixed precision

In [ ]:
model=UNet().to(DEVICE)
opt=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=20)
scaler=torch.cuda.amp.GradScaler()
best=0.0
for epoch in range(20):
    model.train(); tr=[]
    for x,y in train_dl:
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(): loss=loss_fn(model(x),y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tr.append(float(loss))
    model.eval(); va=[]
    with torch.no_grad():
        for x,y in valid_dl:
            x=x.to(DEVICE); y=y.to(DEVICE); z=model(x); va.append(dice_metric(z,y))
    score=float(np.mean(va)); sched.step()
    print(f'epoch {epoch+1:02d} loss={np.mean(tr):.4f} val_dice={score:.4f}')
    if score>best:
        best=score; torch.save(model.state_dict(),OUT/'unet_best.pt')
print('best validation Dice:',best)

## 4. Full-resolution inference and RLE submission

In [ ]:
model.load_state_dict(torch.load(OUT/'unet_best.pt',map_location=DEVICE)); model.eval()
@torch.no_grad()
def predict_full(path):
    x=np.asarray(Image.open(path).convert('L'),np.float32)/255.0
    small=np.asarray(Image.fromarray((x*255).astype(np.uint8)).resize((IMG_SIZE,IMG_SIZE),Image.BILINEAR),np.float32)/255.0
    z=model(torch.from_numpy(small[None,None]).to(DEVICE))
    p=torch.sigmoid(z).float().cpu().numpy()[0,0]
    return np.asarray(Image.fromarray((p*255).astype(np.uint8)).resize((2048,2048),Image.BILINEAR),np.float32)/255.0

def rle(mask):
    z=mask_utils.encode(np.asfortranarray(mask.astype(np.uint8))); c=z['counts']; return c.decode() if isinstance(c,bytes) else c
rows=[]
for k,path in enumerate(sorted(TEST_DIR.glob('*.jpeg'))):
    prob=predict_full(path); binary=prob>0.50
    lab=label(binary,connectivity=2); comps=[]
    for reg in regionprops(lab):
        if reg.area>=80: comps.append((reg.area,lab==reg.label))
    comps.sort(reverse=True,key=lambda z:z[0])
    for j,(_,m) in enumerate(comps,1): rows.append({'filament_id':f'{path.stem}_{j}','segmentation_rle':rle(m)})
    if (k+1)%10==0: print(k+1,len(rows))
with (OUT/'submission_unet.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['filament_id','segmentation_rle']); w.writeheader(); w.writerows(rows)
print('wrote',OUT/'submission_unet.csv','rows',len(rows))

## 5. Download the artifacts

No Kaggle submission is made automatically by this notebook. Download `submission_unet.csv` and `unet_best.pt` from the Kaggle notebook Output panel. Copy the notebook itself into a public GitHub repository together with `requirements.txt` and instructions.